# BioNutriGPT — Multi-Modal Risk Architecture

**Colab + NVIDIA T4 implementation**

`Biomarkers → FT-Transformer → Z_bio`  
`Nutrition → SAINT-style Transformer → Z_nutri`  
`Lifestyle + Demographics → FT-Transformer → Z_life`  
`Z_bio + Z_nutri + Z_life → Cross-Modal Transformer → Z_fused`  
`Z_fused → Adaptive Mixture of Experts → Z_MoE`  
`Z_MoE → Diabetes / Obesity / CVD heads → probabilities + MC-Dropout uncertainty`

The notebook reads the CSV from Google Drive and uses only the columns required by each branch.

**Leakage control:** target columns are never inputs. BMI/weight/waist and disease-diagnosis questionnaire variables are excluded from the lifestyle branch.


In [1]:
# 1. Install dependencies
!pip -q install scikit-learn joblib


In [2]:
# 2. Imports + T4
import os, random, math, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [3]:
# 3. Mount Drive and configure paths
from google.colab import drive
drive.mount("/content/drive")

# CHANGE THIS if your CSV is in a different Drive folder.
DATA_PATH = "/content/drive/MyDrive/BioNutriGPT/BioNutriGPT_final_dataset_2005_2018.csv"

OUTPUT_DIR = Path("/content/drive/MyDrive/BioNutriGPT/bionutrigt_multimodal")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_PATH)


Mounted at /content/drive
/content/drive/MyDrive/BioNutriGPT/BioNutriGPT_final_dataset_2005_2018.csv


In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".csv") and "BioNutriGPT" in file:
            print(os.path.join(root, file))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/BioNutriGpt/BioNutriGPT_final_dataset_2005_2018.csv


## Required feature groups

### 1) Biomarkers → FT-Transformer
Laboratory/metabolic measurements.

### 2) Nutrition → SAINT-style Transformer
First-day 24-hour dietary recall variables.

### 3) Lifestyle + Demographics → FT-Transformer
Physical activity + sleep + smoking + alcohol + demographics.

Demographics are **not a fourth encoder**. They are part of `Z_life`.


In [5]:
# 4. Exact columns used by each model

ID_COL = "SEQN"
TARGET_COLS = ["TARGET_DIABETES", "TARGET_OBESITY", "TARGET_CVD"]

# ---------- Biomarkers ----------
BIOMARKER_COLS = [
    "LBXGH", "LBXGLU", "LBXIN", "LBDHDD",
    "LBXTR", "LBDLDL", "LBXAPB", "LBXTC"
]

# ---------- Nutrition ----------
NUTRITION_COLS = [
    "DR1TKCAL", "DR1TPROT", "DR1TCARB", "DR1TSUGR",
    "DR1TFIBE", "DR1TTFAT", "DR1TSFAT", "DR1TMFAT",
    "DR1TPFAT", "DR1TCHOL", "DR1TATOC", "DR1TRET",
    "DR1TVARA", "DR1TACAR", "DR1TBCAR", "DR1TLYCO",
    "DR1TLZ", "DR1TVB1", "DR1TVB2", "DR1TNIAC",
    "DR1TVB6", "DR1TFOLA", "DR1TFA", "DR1TFF",
    "DR1TFDFE", "DR1TVB12", "DR1TVC", "DR1TVK",
    "DR1TCALC", "DR1TPHOS", "DR1TMAGN", "DR1TIRON",
    "DR1TZINC", "DR1TCOPP", "DR1TSODI", "DR1TPOTA",
    "DR1TSELE", "DR1TCAFF", "DR1TALCO"
]

# ---------- Lifestyle ----------
LIFESTYLE_COLS = [
    "PAQ605", "PAQ610", "PAD615",
    "PAQ620", "PAQ625", "PAD630",
    "PAQ635", "PAQ640", "PAD645",
    "PAQ650", "PAQ655", "PAD660",
    "PAQ665", "PAQ670", "PAD675",
    "PAD680", "PAQ710", "PAQ715",

    "SLD010H", "SLD020M", "SLQ030", "SLQ040",
    "SLQ050", "SLQ060", "SLQ120",

    "SMQ020", "SMD030", "SMQ040", "SMQ050Q",
    "SMQ050U", "SMD055", "SMD057",

    "ALQ101", "ALQ110", "ALQ120Q", "ALQ120U",
    "ALQ130", "ALQ140Q", "ALQ140U", "ALQ150",
    "ALQ151"
]

# Demographics are integrated into the lifestyle/context encoder.
DEMO_NUM_COLS = ["RIDAGEYR", "INDHHINC", "INDFMPIR"]
DEMO_CAT_COLS = ["RIAGENDR", "RIDRETH3", "DMDEDUC2", "DMDMARTL"]

ALL_REQUIRED = (
    [ID_COL] + BIOMARKER_COLS + NUTRITION_COLS + LIFESTYLE_COLS +
    DEMO_NUM_COLS + DEMO_CAT_COLS + TARGET_COLS
)

print("Biomarkers:", len(BIOMARKER_COLS))
print("Nutrition:", len(NUTRITION_COLS))
print("Lifestyle:", len(LIFESTYLE_COLS))
print("Demographic numeric:", len(DEMO_NUM_COLS))
print("Demographic categorical:", len(DEMO_CAT_COLS))


Biomarkers: 8
Nutrition: 39
Lifestyle: 41
Demographic numeric: 3
Demographic categorical: 4


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "/content/drive/MyDrive/BioNutriGpt/BioNutriGPT_final_dataset_2005_2018.csv"
)

print("File exists:", DATA_PATH.exists())
print("Path:", DATA_PATH)

# 5. Load only required columns
header = pd.read_csv(DATA_PATH, nrows=0)

available = set(header.columns)

missing = [c for c in ALL_REQUIRED if c not in available]

if missing:
    print("Missing columns:")
    print(missing)
else:
    print("All required columns are available.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
File exists: True
Path: /content/drive/MyDrive/BioNutriGpt/BioNutriGPT_final_dataset_2005_2018.csv
All required columns are available.


In [9]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "/content/drive/MyDrive/BioNutriGpt/BioNutriGPT_final_dataset_2005_2018.csv"
)

# Load the required columns
df = pd.read_csv(DATA_PATH, usecols=ALL_REQUIRED)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print(df.head())

Dataset loaded successfully
Shape: (42143, 99)
      SEQN  RIAGENDR  RIDAGEYR  DMDEDUC2  DMDMARTL  INDHHINC  INDFMPIR  LBXGH  \
0  31130.0       2.0      85.0       4.0       2.0       4.0      1.99    NaN   
1  31131.0       2.0      44.0       4.0       1.0      11.0      4.65    6.0   
2  31132.0       1.0      70.0       5.0       1.0      11.0      5.00    7.1   
3  31134.0       1.0      73.0       3.0       1.0      12.0       NaN    5.9   
4  31136.0       2.0      41.0       4.0       5.0       7.0      3.57    NaN   

   LBXGLU  LBXIN  ...  PAQ670  PAD675  PAD680  RIDRETH3  PAQ710  PAQ715  \
0     NaN    NaN  ...     NaN     NaN     NaN       NaN     NaN     NaN   
1    90.0  10.03  ...     NaN     NaN     NaN       NaN     NaN     NaN   
2   157.0   8.99  ...     NaN     NaN     NaN       NaN     NaN     NaN   
3   100.0  14.51  ...     NaN     NaN     NaN       NaN     NaN     NaN   
4     NaN    NaN  ...     NaN     NaN     NaN       NaN     NaN     NaN   

   ALQ151  TARG

In [10]:
# 6. Feature engineering
NUM_COLS = BIOMARKER_COLS + NUTRITION_COLS + LIFESTYLE_COLS + DEMO_NUM_COLS

for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in DEMO_CAT_COLS:
    df[c] = df[c].astype("string")

def add_lifestyle_features(frame):
    x = frame.copy()
    x["SLEEP_HOURS_DERIVED"] = x["SLD010H"] + x["SLD020M"] / 60.0

    pairs = [
        ("PAQ610", "PAD615"),
        ("PAQ625", "PAD630"),
        ("PAQ640", "PAD645"),
        ("PAQ655", "PAD660"),
        ("PAQ670", "PAD675"),
    ]

    names = []
    for f, m in pairs:
        name = f"{f}_{m}_MIN"
        x[name] = pd.to_numeric(x[f], errors="coerce") * pd.to_numeric(x[m], errors="coerce")
        names.append(name)

    x["TOTAL_ACTIVITY_MIN_DERIVED"] = x[names].sum(axis=1, min_count=1)
    return x

df = add_lifestyle_features(df)

LIFESTYLE_NUM_COLS = LIFESTYLE_COLS + [
    "SLEEP_HOURS_DERIVED",
    "PAQ610_PAD615_MIN",
    "PAQ625_PAD630_MIN",
    "PAQ640_PAD645_MIN",
    "PAQ655_PAD660_MIN",
    "PAQ670_PAD675_MIN",
    "TOTAL_ACTIVITY_MIN_DERIVED"
]

print("Lifestyle numeric:", len(LIFESTYLE_NUM_COLS))


Lifestyle numeric: 48


In [12]:
# 7. Train / validation / test split

# Remove rows where any target value is missing
before = len(df)

df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

print(f"Removed {before - len(df)} rows with missing target values")
print(f"Remaining rows: {len(df)}")

# Make sure targets are valid integers
stratify_code = (
    df[TARGET_COLS]
    .astype(int)
    .astype(str)
    .agg("".join, axis=1)
)

# First split: 70% train, 30% temporary
train_idx, temp_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.30,
    random_state=42,
    stratify=stratify_code
)

# Split temporary into 15% validation + 15% test
temp_stratify = stratify_code.iloc[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_stratify
)

print("\nSplit sizes:")
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))


Removed 5414 rows with missing target values
Remaining rows: 36729

Split sizes:
Train: 25710
Validation: 5509
Test: 5510


In [14]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

# Prepare categorical data
cat_data = df[DEMO_CAT_COLS].copy()

# Convert pandas StringDtype / pd.NA to normal object + np.nan
for c in DEMO_CAT_COLS:
    cat_data[c] = cat_data[c].astype(object)
    cat_data[c] = cat_data[c].where(pd.notna(cat_data[c]), np.nan)

# Split categorical data
cat_train = cat_data.iloc[train_idx]
cat_val = cat_data.iloc[val_idx]
cat_test = cat_data.iloc[test_idx]

# Impute missing categorical values
cat_imp = SimpleImputer(
    strategy="most_frequent"
)

cat_train_imp = cat_imp.fit_transform(cat_train)
cat_val_imp = cat_imp.transform(cat_val)
cat_test_imp = cat_imp.transform(cat_test)

# Encode categories
cat_enc = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

cat_train_encoded = cat_enc.fit_transform(cat_train_imp)
cat_val_encoded = cat_enc.transform(cat_val_imp)
cat_test_encoded = cat_enc.transform(cat_test_imp)

print("Categorical preprocessing complete")
print("Train:", cat_train_encoded.shape)
print("Validation:", cat_val_encoded.shape)
print("Test:", cat_test_encoded.shape)


Categorical preprocessing complete
Train: (25710, 4)
Validation: (5509, 4)
Test: (5510, 4)


In [15]:
# 9. Dataset/DataLoaders
class MultiModalDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        return {
            "bio": torch.tensor(bio_x[idx], dtype=torch.float32),
            "bio_missing": torch.tensor(bio_missing[idx], dtype=torch.float32),
            "nutri": torch.tensor(nutri_x[idx], dtype=torch.float32),
            "nutri_missing": torch.tensor(nutri_missing[idx], dtype=torch.float32),
            "life": torch.tensor(life_x[idx], dtype=torch.float32),
            "life_missing": torch.tensor(life_missing[idx], dtype=torch.float32),
            "life_cat": torch.tensor(demo_cat_x[idx], dtype=torch.long),
            "y": torch.tensor(y[idx], dtype=torch.float32),
        }

BATCH_SIZE = 512

train_loader = DataLoader(
    MultiModalDataset(train_idx), batch_size=BATCH_SIZE,
    shuffle=True, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    MultiModalDataset(val_idx), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    MultiModalDataset(test_idx), batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)


## Model blocks

The implementation below follows the requested architecture:

- **FT-Transformer** for biomarkers
- **SAINT-style tabular Transformer** for nutrition
- **FT-Transformer** for lifestyle + demographics
- **Cross-Modal Transformer** with multi-head attention
- **Adaptive MoE** with Metabolic/Nutrition/Lifestyle experts and a gating network
- **Three task heads**
- **MC-Dropout uncertainty**


In [16]:
# 10. Tokenizers and FT-Transformer

class NumericalTokenizer(nn.Module):
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x):
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)

class MissingnessEmbedding(nn.Module):
    def __init__(self, d_token):
        super().__init__()
        self.emb = nn.Embedding(2, d_token)

    def forward(self, missing):
        return self.emb(missing.long())

class CategoricalTokenizer(nn.Module):
    def __init__(self, cardinalities, d_token):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(int(card) + 1, d_token) for card in cardinalities
        ])

    def forward(self, x):
        return torch.cat(
            [emb(x[:, j]).unsqueeze(1) for j, emb in enumerate(self.embeddings)],
            dim=1
        )

class FTTransformerEncoder(nn.Module):
    def __init__(self, n_num, n_cat=0, cat_cardinalities=None,
                 d_token=64, n_heads=4, n_layers=3,
                 ff_dim=256, out_dim=128, dropout=0.15):
        super().__init__()
        self.num_tokenizer = NumericalTokenizer(n_num, d_token)
        self.num_missing = MissingnessEmbedding(d_token)
        self.n_cat = n_cat

        if n_cat:
            self.cat_tokenizer = CategoricalTokenizer(cat_cardinalities, d_token)

        self.cls = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.normal_(self.cls, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_token)
        self.proj = nn.Sequential(
            nn.Linear(d_token, out_dim), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, x_num, x_missing, x_cat=None):
        tokens = self.num_tokenizer(x_num) + self.num_missing(x_missing)

        if self.n_cat:
            tokens = torch.cat([tokens, self.cat_tokenizer(x_cat)], dim=1)

        cls = self.cls.expand(x_num.size(0), -1, -1)
        h = torch.cat([cls, tokens], dim=1)
        h = self.encoder(h)

        return self.proj(self.norm(h[:, 0]))


In [17]:
# 11. SAINT-style nutrition encoder

class SAINTStyleEncoder(nn.Module):
    def __init__(self, n_features, d_token=64, n_heads=4,
                 n_layers=3, ff_dim=256, out_dim=128, dropout=0.15):
        super().__init__()
        self.tokenizer = NumericalTokenizer(n_features, d_token)
        self.missing = MissingnessEmbedding(d_token)

        layer = nn.TransformerEncoderLayer(
            d_model=d_token, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.col_encoder = nn.TransformerEncoder(layer, num_layers=n_layers)

        self.row_mixer = nn.Sequential(
            nn.Linear(n_features * d_token, ff_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_dim, n_features * d_token)
        )

        self.norm = nn.LayerNorm(d_token)
        self.proj = nn.Sequential(
            nn.Linear(d_token, out_dim), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, x, missing):
        h = self.tokenizer(x) + self.missing(missing)
        h = self.col_encoder(h)

        # Row-wise context mixing keeps the model T4-friendly.
        mixed = self.row_mixer(h.flatten(1)).view_as(h)
        h = h + mixed

        return self.proj(self.norm(h.mean(dim=1)))


In [18]:
# 12. Cross-Modal Transformer

class CrossModalTransformer(nn.Module):
    def __init__(self, d_in=128, d_model=256, n_heads=8, n_layers=2, dropout=0.15):
        super().__init__()
        self.proj = nn.Linear(d_in, d_model)
        self.modality_embedding = nn.Parameter(torch.randn(1, 3, d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=4*d_model, dropout=dropout,
            batch_first=True, norm_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z_bio, z_nutri, z_life):
        x = torch.stack([z_bio, z_nutri, z_life], dim=1)
        x = self.proj(x) + self.modality_embedding
        x = self.encoder(x)

        # Modality-aware pooling
        z_fused = self.norm(x.mean(dim=1))
        return z_fused, x


In [19]:
# 13. Adaptive Mixture of Experts

class Expert(nn.Module):
    def __init__(self, d_in=256, d_hidden=256, d_out=256, dropout=0.15):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_hidden, d_out), nn.GELU()
        )

    def forward(self, x):
        return self.net(x)

class AdaptiveMoE(nn.Module):
    def __init__(self, d_in=256, d_out=256, dropout=0.15):
        super().__init__()
        self.experts = nn.ModuleList([
            Expert(d_in, 256, d_out, dropout),
            Expert(d_in, 256, d_out, dropout),
            Expert(d_in, 256, d_out, dropout),
        ])

        self.gate = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, 128), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 3)
        )
        self.norm = nn.LayerNorm(d_out)

    def forward(self, z_fused):
        experts = torch.stack([e(z_fused) for e in self.experts], dim=1)
        weights = torch.softmax(self.gate(z_fused), dim=-1)
        z_moe = (experts * weights.unsqueeze(-1)).sum(dim=1)
        return self.norm(z_moe), weights


In [20]:
# 14. Complete BioNutriGPT model

class BioNutriGPT(nn.Module):
    def __init__(self, n_bio, n_nutri, n_life_num,
                 life_cat_cardinalities, d=128, fused_dim=256, dropout=0.15):
        super().__init__()

        self.bio_encoder = FTTransformerEncoder(
            n_num=n_bio, d_token=64, n_heads=4, n_layers=3,
            ff_dim=256, out_dim=d, dropout=dropout
        )

        self.nutri_encoder = SAINTStyleEncoder(
            n_features=n_nutri, d_token=64, n_heads=4, n_layers=3,
            ff_dim=256, out_dim=d, dropout=dropout
        )

        self.life_encoder = FTTransformerEncoder(
            n_num=n_life_num,
            n_cat=len(life_cat_cardinalities),
            cat_cardinalities=life_cat_cardinalities,
            d_token=64, n_heads=4, n_layers=3,
            ff_dim=256, out_dim=d, dropout=dropout
        )

        self.cross_modal = CrossModalTransformer(
            d_in=d, d_model=fused_dim, n_heads=8, n_layers=2, dropout=dropout
        )

        self.moe = AdaptiveMoE(fused_dim, fused_dim, dropout)

        self.diabetes_head = nn.Linear(fused_dim, 1)
        self.obesity_head = nn.Linear(fused_dim, 1)
        self.cvd_head = nn.Linear(fused_dim, 1)

    def forward(self, b):
        z_bio = self.bio_encoder(b["bio"], b["bio_missing"])
        z_nutri = self.nutri_encoder(b["nutri"], b["nutri_missing"])
        z_life = self.life_encoder(b["life"], b["life_missing"], b["life_cat"])

        z_fused, modality_tokens = self.cross_modal(z_bio, z_nutri, z_life)
        z_moe, gate_weights = self.moe(z_fused)

        logits = torch.cat([
            self.diabetes_head(z_moe),
            self.obesity_head(z_moe),
            self.cvd_head(z_moe)
        ], dim=1)

        return {
            "z_bio": z_bio, "z_nutri": z_nutri, "z_life": z_life,
            "z_fused": z_fused, "z_moe": z_moe,
            "modality_tokens": modality_tokens,
            "gate_weights": gate_weights,
            "logits": logits
        }


In [21]:
# 15. Instantiate model
life_cat_cardinalities = [int(len(c)) + 1 for c in cat_enc.categories_]

model = BioNutriGPT(
    n_bio=len(BIOMARKER_COLS),
    n_nutri=len(NUTRITION_COLS),
    n_life_num=life_x.shape[1],
    life_cat_cardinalities=life_cat_cardinalities,
    d=128, fused_dim=256, dropout=0.15
).to(device)

print("Parameters:", round(sum(p.numel() for p in model.parameters())/1e6, 2), "M")


Parameters: 3.81 M


In [23]:
# Create target matrix
y = df[TARGET_COLS].astype(np.float32).values

print("y shape:", y.shape)
print("Target columns:", TARGET_COLS)

y shape: (36729, 3)
Target columns: ['TARGET_DIABETES', 'TARGET_OBESITY', 'TARGET_CVD']


In [26]:
y_train = y[train_idx]

pos = y_train.sum(axis=0)
neg = len(y_train) - pos

pos_weight = np.divide(
    neg,
    pos,
    out=np.ones_like(pos, dtype=np.float32),
    where=pos > 0
)

pos_weight = torch.tensor(
    pos_weight,
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

print("pos_weight:", pos_weight.detach().cpu().numpy())


pos_weight: [6.640416  1.6573644 9.404695 ]


In [30]:
from pathlib import Path

MODEL_DIR = Path(
    "/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_path = MODEL_DIR / "best_bionutrigt_multimodal.pt"

print("Best model will be saved to:")
print(best_path)

Best model will be saved to:
/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/best_bionutrigt_multimodal.pt


In [36]:
# Create full categorical feature array

demo_cat_x = np.zeros(
    (len(df), len(DEMO_CAT_COLS)),
    dtype=np.int64
)

demo_cat_x[train_idx] = cat_train_encoded.astype(np.int64)
demo_cat_x[val_idx] = cat_val_encoded.astype(np.int64)
demo_cat_x[test_idx] = cat_test_encoded.astype(np.int64)

print("demo_cat_x shape:", demo_cat_x.shape)

demo_cat_x shape: (36729, 4)


In [38]:
import torch
from torch.utils.data import Dataset, DataLoader

class BioDataset(Dataset):

    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]

        return {
            "life_num": torch.tensor(
                demo_num_x[i],
                dtype=torch.float32
            ),

            "life_cat": torch.tensor(
                demo_cat_x[i],
                dtype=torch.long
            ),

            "y": torch.tensor(
                y[i],
                dtype=torch.float32
            )
        }

print("BioDataset defined successfully")

BioDataset defined successfully


In [55]:
# train_dataset = BioDataset(train_idx)
# val_dataset = BioDataset(val_idx)
# test_dataset = BioDataset(test_idx)

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=0
# )

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=0
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=0
# )

# print("DataLoaders recreated successfully")

import torch
from torch.utils.data import Dataset, DataLoader

BIO_COLS = [
    'LBXGH','LBXGLU','LBXIN','LBDHDD','LBXTR','LBDLDL','LBXAPB','LBXTC'
]

NUTRI_COLS = [
    'DR1TKCAL','DR1TPROT','DR1TCARB','DR1TSUGR','DR1TFIBE','DR1TTFAT',
    'DR1TSFAT','DR1TMFAT','DR1TPFAT','DR1TCHOL','DR1TATOC','DR1TRET',
    'DR1TVARA','DR1TACAR','DR1TBCAR','DR1TLYCO','DR1TLZ','DR1TVB1',
    'DR1TVB2','DR1TNIAC','DR1TVB6','DR1TFOLA','DR1TFA','DR1TFF',
    'DR1TFDFE','DR1TVB12','DR1TVC','DR1TVK','DR1TCALC','DR1TPHOS',
    'DR1TMAGN','DR1TIRON','DR1TZINC','DR1TCOPP','DR1TSODI','DR1TPOTA',
    'DR1TSELE','DR1TCAFF','DR1TALCO'
]

LIFE_COLS = [
    'PAQ605','PAQ610','PAD615','PAQ620','PAQ625','PAD630','PAQ635','PAQ640',
    'PAD645','PAQ650','PAQ655','PAD660','PAQ665','PAQ670','PAD675','PAD680',
    'PAQ710','PAQ715','SLD010H','SLD020M','SLQ030','SLQ040','SLQ050',
    'SLQ060','SLQ120','SMQ020','SMD030','SMQ040','SMQ050Q','SMQ050U',
    'SMD055','SMD057','ALQ101','ALQ110','ALQ120Q','ALQ120U','ALQ130',
    'ALQ140Q','ALQ140U','ALQ150','ALQ151','RIDAGEYR','INDHHINC','INDFMPIR'
]

LIFE_CAT_COLS = [
    'RIAGENDR',
    'RIDRETH3',
    'DMDEDUC2',
    'DMDMARTL'
]

class MultiModalDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]

        bio = torch.tensor(
            bio_x[idx],
            dtype=torch.float32
        )

        nutri = torch.tensor(
            nutri_x[idx],
            dtype=torch.float32
        )

        life = torch.tensor(
            life_x[idx],
            dtype=torch.float32
        )

        life_cat = torch.tensor(
            demo_cat_x[idx],
            dtype=torch.long
        )

        bio_missing = torch.tensor(
            np.isnan(bio_x[idx]),
            dtype=torch.long
        )

        nutri_missing = torch.tensor(
            np.isnan(nutri_x[idx]),
            dtype=torch.long
        )

        life_missing = torch.tensor(
            np.isnan(life_x[idx]),
            dtype=torch.long
        )

        target = torch.tensor(
            y[idx],
            dtype=torch.float32
        )

        return {
            "bio": bio,
            "bio_missing": bio_missing,
            "nutri": nutri,
            "nutri_missing": nutri_missing,
            "life": life,
            "life_missing": life_missing,
            "life_cat": life_cat,
            "y": target
        }


train_dataset = MultiModalDataset(train_idx)
val_dataset = MultiModalDataset(val_idx)
test_dataset = MultiModalDataset(test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Multimodal DataLoaders created.")

Multimodal DataLoaders created.


In [57]:
batch = next(iter(train_loader))

print(batch.keys())

for k, v in batch.items():
    print(k, v.shape, v.dtype)

dict_keys(['bio', 'bio_missing', 'nutri', 'nutri_missing', 'life', 'life_missing', 'life_cat', 'y'])
bio torch.Size([512, 8]) torch.float32
bio_missing torch.Size([512, 8]) torch.int64
nutri torch.Size([512, 39]) torch.float32
nutri_missing torch.Size([512, 39]) torch.int64
life torch.Size([512, 48]) torch.float32
life_missing torch.Size([512, 48]) torch.int64
life_cat torch.Size([512, 4]) torch.int64
y torch.Size([512, 3]) torch.float32


In [58]:
import inspect

print(inspect.signature(model.forward))

(b)


In [59]:
import inspect

print(inspect.signature(model.forward))

(b)


In [60]:
import inspect

print(inspect.getsource(model.forward))

    def forward(self, b):
        z_bio = self.bio_encoder(b["bio"], b["bio_missing"])
        z_nutri = self.nutri_encoder(b["nutri"], b["nutri_missing"])
        z_life = self.life_encoder(b["life"], b["life_missing"], b["life_cat"])

        z_fused, modality_tokens = self.cross_modal(z_bio, z_nutri, z_life)
        z_moe, gate_weights = self.moe(z_fused)

        logits = torch.cat([
            self.diabetes_head(z_moe),
            self.obesity_head(z_moe),
            self.cvd_head(z_moe)
        ], dim=1)

        return {
            "z_bio": z_bio, "z_nutri": z_nutri, "z_life": z_life,
            "z_fused": z_fused, "z_moe": z_moe,
            "modality_tokens": modality_tokens,
            "gate_weights": gate_weights,
            "logits": logits
        }



In [61]:
with torch.no_grad():
    batch_device = {
        k: v.to(device)
        for k, v in batch.items()
    }

    output = model(batch_device)

print("Forward pass successful")
print("Logits shape:", output["logits"].shape)

Forward pass successful
Logits shape: torch.Size([512, 3])


In [62]:
# 17. Train/evaluate
# 17. Train / evaluate

# from sklearn.metrics import roc_auc_score
# import numpy as np
# import torch

# def run_epoch(loader, train=True):
#     if train:
#         model.train()
#     else:
#         model.eval()

#     total_loss = 0.0
#     n = 0

#     all_p = []
#     all_y = []

#     for batch in loader:

#         batch = {
#             k: v.to(device, non_blocking=True)
#             for k, v in batch.items()
#         }

#         if train:
#             optimizer.zero_grad(set_to_none=True)

#         with torch.set_grad_enabled(train):

#             outputs = model(
#                 life_num=batch["life_num"],
#                 life_cat=batch["life_cat"]
#             )

#             loss = criterion(outputs, batch["y"])

#             if train:
#                 loss.backward()
#                 optimizer.step()

#         bs = batch["y"].size(0)

#         total_loss += loss.item() * bs
#         n += bs

#         probs = torch.sigmoid(outputs)

#         all_p.append(probs.detach().cpu().numpy())
#         all_y.append(batch["y"].detach().cpu().numpy())

#     all_p = np.concatenate(all_p)
#     all_y = np.concatenate(all_y)

#     avg_loss = total_loss / n

#     aucs = []

#     for j in range(all_y.shape[1]):
#         try:
#             auc = roc_auc_score(
#                 all_y[:, j],
#                 all_p[:, j]
#             )
#         except ValueError:
# 17A. Train / Evaluate Epoch

from sklearn.metrics import roc_auc_score
import numpy as np
import torch


def run_epoch(loader, train=True):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    n = 0

    all_p = []
    all_y = []

    for batch in loader:

        # Move tensors to device
        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):

            # BioNutriGPT expects ONE dictionary
            output = model(batch)

            # Model returns logits inside dictionary
            logits = output["logits"]

            # Multi-label BCE loss
            loss = criterion(
                logits,
                batch["y"]
            )

            if train:
                loss.backward()
                optimizer.step()

        # Batch size
        bs = batch["y"].size(0)

        total_loss += loss.item() * bs
        n += bs

        # Convert logits → probabilities
        probs = torch.sigmoid(logits)

        all_p.append(
            probs.detach().cpu().numpy()
        )

        all_y.append(
            batch["y"].detach().cpu().numpy()
        )

    # Combine all batches
    all_p = np.concatenate(all_p, axis=0)
    all_y = np.concatenate(all_y, axis=0)

    # Average loss
    avg_loss = total_loss / n

    # AUC for each target
    aucs = []

    for j in range(all_y.shape[1]):

        try:
            auc = roc_auc_score(
                all_y[:, j],
                all_p[:, j]
            )
        except ValueError:
            auc = np.nan

        aucs.append(auc)

    # Mean AUC
    mean_auc = np.nanmean(aucs)

    return (
        avg_loss,
        mean_auc,
        aucs,
        all_p,
        all_y
    )


print("run_epoch() updated successfully")

run_epoch() updated successfully


In [63]:
batch = next(iter(train_loader))
print(batch.keys())

for k, v in batch.items():
    print(k, v.shape, v.dtype)

dict_keys(['bio', 'bio_missing', 'nutri', 'nutri_missing', 'life', 'life_missing', 'life_cat', 'y'])
bio torch.Size([512, 8]) torch.float32
bio_missing torch.Size([512, 8]) torch.int64
nutri torch.Size([512, 39]) torch.float32
nutri_missing torch.Size([512, 39]) torch.int64
life torch.Size([512, 48]) torch.float32
life_missing torch.Size([512, 48]) torch.int64
life_cat torch.Size([512, 4]) torch.int64
y torch.Size([512, 3]) torch.float32


In [64]:
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_auc, _, _, _ = run_epoch(train_loader, train=True)

    va_loss, va_auc, va_aucs, _, _ = run_epoch(
        val_loader,
        train=False
    )

    scheduler.step(va_auc)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {tr_loss:.4f} | "
        f"Train AUC: {tr_auc:.4f} | "
        f"Val Loss: {va_loss:.4f} | "
        f"Val AUC: {va_auc:.4f}"
    )

    print(
        f"  Diabetes: {va_aucs[0]:.4f} | "
        f"Obesity: {va_aucs[1]:.4f} | "
        f"CVD: {va_aucs[2]:.4f}"
    )

    if va_auc > best_val_auc:
        best_val_auc = va_auc

        torch.save(
            {
                "model_state": model.state_dict(),
                "best_val_auc": best_val_auc,
                "epoch": epoch
            },
            str(best_path)
        )

        print("  ✓ Best model saved")

Epoch 01 | Train Loss: 1.1312 | Train AUC: 0.5025 | Val Loss: 1.1097 | Val AUC: 0.5980
  Diabetes: 0.7009 | Obesity: 0.5488 | CVD: 0.5444
  ✓ Best model saved
Epoch 02 | Train Loss: 1.1128 | Train AUC: 0.4945 | Val Loss: 1.1080 | Val AUC: 0.6465
  Diabetes: 0.8041 | Obesity: 0.5385 | CVD: 0.5969
  ✓ Best model saved
Epoch 03 | Train Loss: 1.1105 | Train AUC: 0.5014 | Val Loss: 1.1096 | Val AUC: 0.6777
  Diabetes: 0.8453 | Obesity: 0.5279 | CVD: 0.6598
  ✓ Best model saved
Epoch 04 | Train Loss: 1.1058 | Train AUC: 0.5200 | Val Loss: 1.0492 | Val AUC: 0.7300
  Diabetes: 0.8540 | Obesity: 0.6453 | CVD: 0.6905
  ✓ Best model saved
Epoch 05 | Train Loss: 0.9968 | Train AUC: 0.6741 | Val Loss: 0.9363 | Val AUC: 0.7599
  Diabetes: 0.8946 | Obesity: 0.6555 | CVD: 0.7297
  ✓ Best model saved
Epoch 06 | Train Loss: 0.9030 | Train AUC: 0.7365 | Val Loss: 0.8749 | Val AUC: 0.7700
  Diabetes: 0.9114 | Obesity: 0.6606 | CVD: 0.7380
  ✓ Best model saved
Epoch 07 | Train Loss: 0.8643 | Train AUC: 0.7

In [49]:
print("=" * 60)
print("MODEL ARCHITECTURE")
print("=" * 60)

print(model)

print("\n" + "=" * 60)
print("MODEL PARAMETERS / ATTRIBUTES")
print("=" * 60)

print("Bio encoder:", model.bio_encoder)
print("Nutri encoder:", model.nutri_encoder)
print("Life encoder:", model.life_encoder)

print("\n" + "=" * 60)
print("CURRENT BATCH KEYS")
print("=" * 60)

print(list(batch.keys()))

MODEL ARCHITECTURE
BioNutriGPT(
  (bio_encoder): FTTransformerEncoder(
    (num_tokenizer): NumericalTokenizer()
    (num_missing): MissingnessEmbedding(
      (emb): Embedding(2, 64)
    )
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.15, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.15, inplace=False)
          (dropout2): Dropout(p=0.15, inplace=False)
        )
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (proj):

In [65]:
import inspect

print(inspect.signature(model.bio_encoder))
print(inspect.signature(model.nutri_encoder))
print(inspect.signature(model.life_encoder))

(*args, **kwargs)
(*args, **kwargs)
(*args, **kwargs)


In [66]:
print("NUM_COLS:")
print(NUM_COLS)

print("\nDEMO_CAT_COLS:")
print(DEMO_CAT_COLS)

print("\nTARGET_COLS:")
print(TARGET_COLS)

print("\nDataFrame columns:")
print(df.columns.tolist())

NUM_COLS:
['LBXGH', 'LBXGLU', 'LBXIN', 'LBDHDD', 'LBXTR', 'LBDLDL', 'LBXAPB', 'LBXTC', 'DR1TKCAL', 'DR1TPROT', 'DR1TCARB', 'DR1TSUGR', 'DR1TFIBE', 'DR1TTFAT', 'DR1TSFAT', 'DR1TMFAT', 'DR1TPFAT', 'DR1TCHOL', 'DR1TATOC', 'DR1TRET', 'DR1TVARA', 'DR1TACAR', 'DR1TBCAR', 'DR1TLYCO', 'DR1TLZ', 'DR1TVB1', 'DR1TVB2', 'DR1TNIAC', 'DR1TVB6', 'DR1TFOLA', 'DR1TFA', 'DR1TFF', 'DR1TFDFE', 'DR1TVB12', 'DR1TVC', 'DR1TVK', 'DR1TCALC', 'DR1TPHOS', 'DR1TMAGN', 'DR1TIRON', 'DR1TZINC', 'DR1TCOPP', 'DR1TSODI', 'DR1TPOTA', 'DR1TSELE', 'DR1TCAFF', 'DR1TALCO', 'PAQ605', 'PAQ610', 'PAD615', 'PAQ620', 'PAQ625', 'PAD630', 'PAQ635', 'PAQ640', 'PAD645', 'PAQ650', 'PAQ655', 'PAD660', 'PAQ665', 'PAQ670', 'PAD675', 'PAD680', 'PAQ710', 'PAQ715', 'SLD010H', 'SLD020M', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ060', 'SLQ120', 'SMQ020', 'SMD030', 'SMQ040', 'SMQ050Q', 'SMQ050U', 'SMD055', 'SMD057', 'ALQ101', 'ALQ110', 'ALQ120Q', 'ALQ120U', 'ALQ130', 'ALQ140Q', 'ALQ140U', 'ALQ150', 'ALQ151', 'RIDAGEYR', 'INDHHINC', 'INDFMPIR']

DEMO

In [67]:
from pathlib import Path

print("Checkpoint exists:", best_path.exists())
print("Checkpoint path:", best_path)

Checkpoint exists: True
Checkpoint path: /content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/best_bionutrigt_multimodal.pt


In [69]:
# # 18. Test metrics
# ckpt = torch.load(best_path, map_location=device)
# model.load_state_dict(ckpt["model_state"])

# test_loss, test_auc, test_aucs, test_p, test_y = run_epoch(test_loader, False)

# print("Test loss:", test_loss)
# print("Mean ROC-AUC:", test_auc)
# for name, auc in zip(TARGET_COLS, test_aucs):
#     print(name, "ROC-AUC:", round(float(auc), 4))
# 18. Test metrics

ckpt = torch.load(
    best_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(ckpt["model_state"])

test_loss, test_auc, test_aucs, test_p, test_y = run_epoch(
    test_loader,
    train=False
)

print("Test loss:", test_loss)
print("Mean ROC-AUC:", test_auc)

for name, auc in zip(TARGET_COLS, test_aucs):
    print(name, "ROC-AUC:", round(float(auc), 4))


Test loss: 0.7856663467014333
Mean ROC-AUC: 0.8069189964052431
TARGET_DIABETES ROC-AUC: 0.929
TARGET_OBESITY ROC-AUC: 0.6867
TARGET_CVD ROC-AUC: 0.8051


In [70]:
from pathlib import Path

base = Path("/content/drive/MyDrive")

matches = list(base.rglob("*.pt"))

print("Checkpoint files found:")

for p in matches:
    print(p)

Checkpoint files found:
/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/best_bionutrigt_multimodal.pt


In [71]:
from pathlib import Path

MODEL_DIR = Path(
    "/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_path = MODEL_DIR / "best_bionutrigt_multimodal.pt"

best_val_auc = -np.inf

print("Checkpoint path:")
print(best_path)

Checkpoint path:
/content/drive/MyDrive/BioNutriGpt/bionutrigt_multimodal/best_bionutrigt_multimodal.pt


## MC-Dropout uncertainty

For each test sample, the model is run multiple times with dropout enabled.

For each task:

- `mean probability` = prediction
- `standard deviation` = uncertainty
- `2.5%–97.5%` = empirical uncertainty interval


In [72]:
# 19. MC-Dropout uncertainty
def enable_dropout(m):
    if isinstance(m, nn.Dropout):
        m.train()

@torch.no_grad()
def mc_predict(loader, passes=30):
    model.eval()
    model.apply(enable_dropout)

    samples = []

    for _ in range(passes):
        preds = []
        for batch in loader:
            batch = {k:v.to(device, non_blocking=True) for k,v in batch.items()}
            out = model(batch)
            preds.append(torch.sigmoid(out["logits"]).cpu().numpy())
        samples.append(np.concatenate(preds))

    samples = np.stack(samples)
    mean = samples.mean(axis=0)
    std = samples.std(axis=0)
    lo = np.percentile(samples, 2.5, axis=0)
    hi = np.percentile(samples, 97.5, axis=0)

    return mean, std, lo, hi

mc_mean, mc_std, mc_lo, mc_hi = mc_predict(test_loader, passes=30)

for j, name in enumerate(["DIABETES", "OBESITY", "CVD"]):
    print(
        name,
        "mean probability =", round(float(mc_mean[:,j].mean()), 4),
        "mean uncertainty =", round(float(mc_std[:,j].mean()), 4)
    )

model.eval()


DIABETES mean probability = 0.2751 mean uncertainty = 0.0179
OBESITY mean probability = 0.4452 mean uncertainty = 0.0225
CVD mean probability = 0.3643 mean uncertainty = 0.0217


BioNutriGPT(
  (bio_encoder): FTTransformerEncoder(
    (num_tokenizer): NumericalTokenizer()
    (num_missing): MissingnessEmbedding(
      (emb): Embedding(2, 64)
    )
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
          )
          (linear1): Linear(in_features=64, out_features=256, bias=True)
          (dropout): Dropout(p=0.15, inplace=False)
          (linear2): Linear(in_features=256, out_features=64, bias=True)
          (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.15, inplace=False)
          (dropout2): Dropout(p=0.15, inplace=False)
        )
      )
    )
    (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (proj): Sequential(
      

In [73]:
# 20. Save risk predictions + uncertainty
test_ids = df.iloc[test_idx][ID_COL].values

results = pd.DataFrame({
    "SEQN": test_ids,

    "DIABETES_PROB": mc_mean[:,0],
    "DIABETES_UNCERTAINTY": mc_std[:,0],
    "DIABETES_LOWER_95": mc_lo[:,0],
    "DIABETES_UPPER_95": mc_hi[:,0],

    "OBESITY_PROB": mc_mean[:,1],
    "OBESITY_UNCERTAINTY": mc_std[:,1],
    "OBESITY_LOWER_95": mc_lo[:,1],
    "OBESITY_UPPER_95": mc_hi[:,1],

    "CVD_PROB": mc_mean[:,2],
    "CVD_UNCERTAINTY": mc_std[:,2],
    "CVD_LOWER_95": mc_lo[:,2],
    "CVD_UPPER_95": mc_hi[:,2],
})

pred_path = OUTPUT_DIR / "risk_predictions_with_uncertainty.csv"
results.to_csv(pred_path, index=False)
print(pred_path)
results.head()


/content/drive/MyDrive/BioNutriGPT/bionutrigt_multimodal/risk_predictions_with_uncertainty.csv


,SEQN,DIABETES_PROB,DIABETES_UNCERTAINTY,DIABETES_LOWER_95,DIABETES_UPPER_95,OBESITY_PROB,OBESITY_UNCERTAINTY,OBESITY_LOWER_95,OBESITY_UPPER_95,CVD_PROB,CVD_UNCERTAINTY,CVD_LOWER_95,CVD_UPPER_95
0,100144.0,0.242822,0.024575,0.203013,0.288800,0.652712,0.019526,0.620842,0.683920,0.252103,0.015850,0.222060,0.282484
1,76169.0,0.026979,0.003460,0.022449,0.034356,0.260749,0.019175,0.231741,0.295153,0.151156,0.014552,0.129779,0.177309
2,77340.0,0.018187,0.004361,0.012956,0.028146,0.250954,0.020164,0.222948,0.300089,0.097572,0.012022,0.078038,0.120704
3,102948.0,0.028565,0.005309,0.021250,0.040847,0.277590,0.023574,0.247406,0.327205,0.140367,0.014011,0.114586,0.165784
4,46525.0,0.088059,0.011087,0.069525,0.110453,0.467365,0.024344,0.425629,0.516221,0.203610,0.013772,0.172929,0.229378


In [74]:
# 21. Extract Z_bio, Z_nutri, Z_life, Z_fused, Z_MoE and gating weights
@torch.no_grad()
def extract_embeddings(loader):
    model.eval()
    rows = []

    for batch in loader:
        b = {k:v.to(device, non_blocking=True) for k,v in batch.items()}
        out = model(b)

        for i in range(b["y"].shape[0]):
            rows.append({
                "z_bio": out["z_bio"][i].cpu().numpy(),
                "z_nutri": out["z_nutri"][i].cpu().numpy(),
                "z_life": out["z_life"][i].cpu().numpy(),
                "z_fused": out["z_fused"][i].cpu().numpy(),
                "z_moe": out["z_moe"][i].cpu().numpy(),
                "gate_metabolic": out["gate_weights"][i,0].item(),
                "gate_nutrition": out["gate_weights"][i,1].item(),
                "gate_lifestyle": out["gate_weights"][i,2].item()
            })
    return rows

rows = extract_embeddings(test_loader)

emb = pd.DataFrame({
    "SEQN": test_ids,
    "gate_metabolic": [r["gate_metabolic"] for r in rows],
    "gate_nutrition": [r["gate_nutrition"] for r in rows],
    "gate_lifestyle": [r["gate_lifestyle"] for r in rows]
})

for key in ["z_bio", "z_nutri", "z_life", "z_fused", "z_moe"]:
    a = np.stack([r[key] for r in rows])
    for j in range(a.shape[1]):
        emb[f"{key}_{j}"] = a[:,j]

emb_path = OUTPUT_DIR / "test_embeddings_and_moe_gates.csv"
emb.to_csv(emb_path, index=False)
print(emb_path)


/content/drive/MyDrive/BioNutriGPT/bionutrigt_multimodal/test_embeddings_and_moe_gates.csv


In [76]:
# 22. Save preprocessing and architecture metadata
import joblib
joblib.dump({
    "bio_imputer": bio_imp, "bio_scaler": bio_scaler,
    "nutri_imputer": nutri_imp, "nutri_scaler": nutri_scaler,
    "life_imputer": life_imp, "life_scaler": life_scaler,
    "demo_imputer": demo_imp, "demo_scaler": demo_scaler,
    "cat_imputer": cat_imp, "cat_encoder": cat_enc,
    "BIOMARKER_COLS": BIOMARKER_COLS,
    "NUTRITION_COLS": NUTRITION_COLS,
    "LIFESTYLE_NUM_COLS": LIFESTYLE_NUM_COLS,
    "DEMO_NUM_COLS": DEMO_NUM_COLS,
    "DEMO_CAT_COLS": DEMO_CAT_COLS,
    "TARGET_COLS": TARGET_COLS
}, OUTPUT_DIR / "preprocessing.joblib")

pd.DataFrame(
    history,
    columns=[
        "epoch", "train_loss", "train_auc",
        "val_loss", "val_auc",
        "val_diabetes_auc", "val_obesity_auc", "val_cvd_auc"
    ]
).to_csv(OUTPUT_DIR / "training_history.csv", index=False)

with open(OUTPUT_DIR / "architecture.json", "w") as f:
    json.dump({
        "biomarker_encoder": "FT-Transformer",
        "nutrition_encoder": "SAINT-style tabular Transformer",
        "lifestyle_demographic_encoder": "FT-Transformer",
        "cross_modal_transformer": "2-layer 8-head Transformer",
        "fusion_dimension": 256,
        "moe_experts": ["Metabolic", "Nutrition", "Lifestyle"],
        "risk_heads": ["Diabetes", "Obesity", "CVD"],
        "uncertainty": "MC-Dropout"
    }, f, indent=2)

print("All artifacts saved in:", OUTPUT_DIR)


All artifacts saved in: /content/drive/MyDrive/BioNutriGPT/bionutrigt_multimodal


In [77]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Convert probabilities to 0/1 predictions
test_pred = (test_p >= 0.5).astype(int)

print("=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

for i, name in enumerate(TARGET_COLS):

    y_true = test_y[:, i]
    y_pred = test_pred[:, i]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{name}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

MODEL PERFORMANCE

TARGET_DIABETES
Accuracy : 0.8826
Precision: 0.5338
Recall   : 0.8100
F1 Score : 0.6435
Confusion Matrix:
[[4279  510]
 [ 137  584]]

TARGET_OBESITY
Accuracy : 0.6574
Precision: 0.5475
Recall   : 0.5164
F1 Score : 0.5315
Confusion Matrix:
[[2551  885]
 [1003 1071]]

TARGET_CVD
Accuracy : 0.7465
Precision: 0.2289
Recall   : 0.6906
F1 Score : 0.3438
Confusion Matrix:
[[3747 1233]
 [ 164  366]]


In [85]:
# ============================================================
# BIO NUTRI GPT — REAL CVD PATIENT TEST
# NO USER INPUT
# ============================================================

import numpy as np
import pandas as pd
import torch
import joblib
from pathlib import Path

print("=" * 70)
print("BIO NUTRI GPT — REAL CVD PATIENT TEST")
print("=" * 70)


# ============================================================
# 1. LOAD PREPROCESSING
# ============================================================

PREP_PATH = Path(
    "/content/drive/MyDrive/BioNutriGPT/"
    "bionutrigt_multimodal/preprocessing.joblib"
)

prep = joblib.load(PREP_PATH)

print("✓ Preprocessing loaded")


# ============================================================
# 2. LOAD PREPROCESSORS
# ============================================================

bio_imputer = prep["bio_imputer"]
bio_scaler = prep["bio_scaler"]

nutri_imputer = prep["nutri_imputer"]
nutri_scaler = prep["nutri_scaler"]

life_imputer = prep["life_imputer"]
life_scaler = prep["life_scaler"]

cat_imputer = prep["cat_imputer"]
cat_encoder = prep["cat_encoder"]

BIOMARKER_COLS = prep["BIOMARKER_COLS"]
NUTRITION_COLS = prep["NUTRITION_COLS"]
LIFESTYLE_NUM_COLS = prep["LIFESTYLE_NUM_COLS"]
DEMO_NUM_COLS = prep["DEMO_NUM_COLS"]
DEMO_CAT_COLS = prep["DEMO_CAT_COLS"]

print("✓ Preprocessors found")


# ============================================================
# 3. REAL PUBLISHED CVD PATIENT
# ============================================================

patient = {}

# ------------------------------------------------------------
# Demographics
# ------------------------------------------------------------

patient["RIDAGEYR"] = 67
patient["RIAGENDR"] = "1.0"       # Male


# ------------------------------------------------------------
# Blood biomarkers
# ------------------------------------------------------------

patient["LBXGH"] = 8.3           # HbA1c %
patient["LBXGLU"] = np.nan       # Not reported
patient["LBXIN"] = np.nan        # Not reported
patient["LBDHDD"] = 28.0         # HDL mg/dL
patient["LBXTR"] = 259.0         # Triglycerides mg/dL
patient["LBDLDL"] = 98.0         # LDL mg/dL
patient["LBXAPB"] = np.nan       # Not inserted
patient["LBXTC"] = np.nan        # Not reported


# ============================================================
# 4. NUTRITION
# ============================================================

for col in NUTRITION_COLS:
    patient[col] = np.nan


# ============================================================
# 5. LIFESTYLE
# ============================================================

# Published case: patient denied smoking
patient["SMQ020"] = "2.0"

for col in LIFESTYLE_NUM_COLS:
    if col not in patient:
        patient[col] = np.nan


# ============================================================
# 6. DEMOGRAPHICS
# ============================================================

for col in DEMO_NUM_COLS:
    if col not in patient:
        patient[col] = np.nan

for col in DEMO_CAT_COLS:
    if col not in patient:
        patient[col] = np.nan


# ============================================================
# 7. DATAFRAME
# ============================================================

patient_df = pd.DataFrame([patient])


# ============================================================
# 8. DERIVED LIFESTYLE FEATURES
# ============================================================

patient_df = add_lifestyle_features(patient_df)


# ============================================================
# 9. ENSURE REQUIRED COLUMNS EXIST
# ============================================================

for col in BIOMARKER_COLS:
    if col not in patient_df.columns:
        patient_df[col] = np.nan

for col in NUTRITION_COLS:
    if col not in patient_df.columns:
        patient_df[col] = np.nan

for col in LIFESTYLE_NUM_COLS:
    if col not in patient_df.columns:
        patient_df[col] = np.nan

for col in DEMO_NUM_COLS:
    if col not in patient_df.columns:
        patient_df[col] = np.nan

for col in DEMO_CAT_COLS:
    if col not in patient_df.columns:
        patient_df[col] = np.nan


# ============================================================
# 10. SELECT INPUTS
# ============================================================

bio_input = patient_df[BIOMARKER_COLS].copy()

nutri_input = patient_df[NUTRITION_COLS].copy()

life_input = patient_df[LIFESTYLE_NUM_COLS].copy()

cat_input = patient_df[DEMO_CAT_COLS].copy()


# ============================================================
# 11. MISSINGNESS MASKS
# ============================================================

bio_missing = (
    bio_input.isna()
    .astype(np.float32)
    .values
)

nutri_missing = (
    nutri_input.isna()
    .astype(np.float32)
    .values
)

life_missing = (
    life_input.isna()
    .astype(np.float32)
    .values
)


# ============================================================
# 12. PREPROCESS
# ============================================================

bio_processed = bio_scaler.transform(
    bio_imputer.transform(bio_input)
)

nutri_processed = nutri_scaler.transform(
    nutri_imputer.transform(nutri_input)
)

life_processed = life_scaler.transform(
    life_imputer.transform(life_input)
)

cat_processed = cat_encoder.transform(
    cat_imputer.transform(cat_input)
)


# ============================================================
# 13. TORCH TENSORS
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

bio_tensor = torch.tensor(
    bio_processed,
    dtype=torch.float32,
    device=device
)

nutri_tensor = torch.tensor(
    nutri_processed,
    dtype=torch.float32,
    device=device
)

life_tensor = torch.tensor(
    life_processed,
    dtype=torch.float32,
    device=device
)

life_cat_tensor = torch.tensor(
    cat_processed,
    dtype=torch.long,
    device=device
)

bio_missing_tensor = torch.tensor(
    bio_missing,
    dtype=torch.float32,
    device=device
)

nutri_missing_tensor = torch.tensor(
    nutri_missing,
    dtype=torch.float32,
    device=device
)

life_missing_tensor = torch.tensor(
    life_missing,
    dtype=torch.float32,
    device=device
)


# ============================================================
# 14. MODEL INPUT
# ============================================================

model_input = {
    "bio": bio_tensor,
    "bio_missing": bio_missing_tensor,

    "nutri": nutri_tensor,
    "nutri_missing": nutri_missing_tensor,

    "life": life_tensor,
    "life_missing": life_missing_tensor,

    "life_cat": life_cat_tensor
}


# ============================================================
# 15. MODEL PREDICTION
# ============================================================

model = model.to(device)
model.eval()

with torch.no_grad():
    output = model(model_input)


# ============================================================
# 16. MODEL OUTPUT STRUCTURE
# ============================================================

print()
print("Model output type:", type(output))

print(
    "Output keys:",
    list(output.keys())
)


# ============================================================
# 17. GET LOGITS
# ============================================================

logits = output["logits"]

print()
print("=" * 70)
print("LOGITS STRUCTURE")
print("=" * 70)

print("Type:", type(logits))

if isinstance(logits, dict):

    print("Logit keys:", list(logits.keys()))

elif isinstance(logits, (list, tuple)):

    print("Number of logits:", len(logits))

    for i, x in enumerate(logits):
        print(
            f"Logit {i}: "
            f"type={type(x)}, "
            f"shape={getattr(x, 'shape', None)}"
        )

else:

    print(
        "Logits shape:",
        getattr(logits, "shape", None)
    )


# ============================================================
# 18. EXTRACT THREE TASK LOGITS
# ============================================================

if isinstance(logits, dict):

    # Try exact target names first
    diabetes_logits = logits.get("TARGET_DIABETES")
    obesity_logits = logits.get("TARGET_OBESITY")
    cvd_logits = logits.get("TARGET_CVD")

    # Try shorter names
    if diabetes_logits is None:
        diabetes_logits = logits.get("diabetes")

    if obesity_logits is None:
        obesity_logits = logits.get("obesity")

    if cvd_logits is None:
        cvd_logits = logits.get("cvd")

    # Try logits naming
    if diabetes_logits is None:
        diabetes_logits = logits.get("diabetes_logits")

    if obesity_logits is None:
        obesity_logits = logits.get("obesity_logits")

    if cvd_logits is None:
        cvd_logits = logits.get("cvd_logits")


elif isinstance(logits, (list, tuple)):

    if len(logits) < 3:
        raise ValueError(
            f"Expected 3 task outputs, "
            f"but found {len(logits)}."
        )

    diabetes_logits = logits[0]
    obesity_logits = logits[1]
    cvd_logits = logits[2]


else:

    # Handle a tensor such as [batch, 3]
    if logits.ndim == 2 and logits.shape[1] >= 3:

        diabetes_logits = logits[:, 0]
        obesity_logits = logits[:, 1]
        cvd_logits = logits[:, 2]

    else:

        raise ValueError(
            "Unable to determine the three task outputs."
        )


# ============================================================
# 19. CHECK EXTRACTION
# ============================================================

print()
print("=" * 70)
print("TASK OUTPUTS")
print("=" * 70)

print(
    "Diabetes logits:",
    getattr(diabetes_logits, "shape", None)
)

print(
    "Obesity logits :",
    getattr(obesity_logits, "shape", None)
)

print(
    "CVD logits     :",
    getattr(cvd_logits, "shape", None)
)


# ============================================================
# 20. PROBABILITY
# ============================================================

def get_probability(x):

    if isinstance(x, (list, tuple)):
        x = x[0]

    if not torch.is_tensor(x):
        x = torch.tensor(x)

    x = x.detach()

    if x.ndim > 1 and x.shape[-1] == 1:
        x = x.squeeze(-1)

    return (
        torch.sigmoid(x)
        .cpu()
        .item()
        * 100
    )


diabetes_prob = get_probability(
    diabetes_logits
)

obesity_prob = get_probability(
    obesity_logits
)

cvd_prob = get_probability(
    cvd_logits
)


# ============================================================
# 21. INPUT COVERAGE
# ============================================================

all_model_cols = (
    BIOMARKER_COLS
    + NUTRITION_COLS
    + LIFESTYLE_NUM_COLS
    + DEMO_NUM_COLS
    + DEMO_CAT_COLS
)

available = int(
    patient_df[all_model_cols]
    .notna()
    .sum(axis=1)
    .iloc[0]
)

total = len(all_model_cols)

coverage = (
    available / total
) * 100


# ============================================================
# 22. RISK LABEL
# ============================================================

def risk_label(prob):

    if prob < 20:
        return "LOW"

    elif prob < 50:
        return "MODERATE"

    else:
        return "HIGH"


# ============================================================
# 23. FINAL RESULT
# ============================================================

print()
print("=" * 70)
print("REAL CVD PATIENT")
print("=" * 70)

print("Age                    : 67")
print("Sex                    : Male")
print("Known CVD              : YES")
print("Clinical event         : Acute myocardial infarction")


print()
print("=" * 70)
print("BLOOD VALUES USED")
print("=" * 70)

print("HbA1c                  : 8.3 %")
print("HDL                    : 28 mg/dL")
print("LDL                    : 98 mg/dL")
print("Triglycerides          : 259 mg/dL")

print()
print("Fasting glucose        : Not reported")
print("Insulin                : Not reported")
print("ApoB                   : Not inserted")
print("Total cholesterol      : Not reported")


print()
print("=" * 70)
print("BIO NUTRI GPT PREDICTION")
print("=" * 70)

print(
    f"Diabetes probability   : "
    f"{diabetes_prob:.2f}%"
)

print(
    f"Obesity probability    : "
    f"{obesity_prob:.2f}%"
)

print(
    f"CVD probability        : "
    f"{cvd_prob:.2f}%"
)


print()
print("=" * 70)
print("MODEL RISK LEVEL")
print("=" * 70)

print(
    f"Diabetes : "
    f"{risk_label(diabetes_prob)}"
)

print(
    f"Obesity  : "
    f"{risk_label(obesity_prob)}"
)

print(
    f"CVD      : "
    f"{risk_label(cvd_prob)}"
)


print()
print("=" * 70)
print("INPUT COVERAGE")
print("=" * 70)

print(
    f"Available model features : "
    f"{available}/{total}"
)

print(
    f"Input coverage           : "
    f"{coverage:.1f}%"
)


print()
print("=" * 70)
print("KNOWN REAL-WORLD OUTCOME")
print("=" * 70)

print("Actual CVD             : YES")
print("Myocardial infarction  : YES")


if cvd_prob >= 50:

    print()
    print("✓ MODEL FLAGGED HIGH CVD RISK")

elif cvd_prob >= 20:

    print()
    print("⚠ MODEL FLAGGED MODERATE CVD RISK")

else:

    print()
    print("✗ MODEL FLAGGED LOW CVD RISK")


print()
print("=" * 70)
print("IMPORTANT")
print("=" * 70)

print(
    "This is a model validation experiment, "
    "not a medical diagnosis."
)

print(
    "The patient has confirmed cardiovascular disease "
    "regardless of the model prediction."
)

print("=" * 70)

BIO NUTRI GPT — REAL CVD PATIENT TEST
✓ Preprocessing loaded
✓ Preprocessors found

Model output type: <class 'dict'>
Output keys: ['z_bio', 'z_nutri', 'z_life', 'z_fused', 'z_moe', 'modality_tokens', 'gate_weights', 'logits']

LOGITS STRUCTURE
Type: <class 'torch.Tensor'>
Logits shape: torch.Size([1, 3])

TASK OUTPUTS
Diabetes logits: torch.Size([1])
Obesity logits : torch.Size([1])
CVD logits     : torch.Size([1])

REAL CVD PATIENT
Age                    : 67
Sex                    : Male
Known CVD              : YES
Clinical event         : Acute myocardial infarction

BLOOD VALUES USED
HbA1c                  : 8.3 %
HDL                    : 28 mg/dL
LDL                    : 98 mg/dL
Triglycerides          : 259 mg/dL

Fasting glucose        : Not reported
Insulin                : Not reported
ApoB                   : Not inserted
Total cholesterol      : Not reported

BIO NUTRI GPT PREDICTION
Diabetes probability   : 97.02%
Obesity probability    : 76.63%
CVD probability        : 8

In [80]:
# ============================================================
# BioNutriGPT — USER FRIENDLY PERSONAL HEALTH PREDICTION
# ============================================================

import numpy as np
import pandas as pd
import torch
import joblib

print("=" * 75)
print("             BioNutriGPT PERSONAL HEALTH ASSESSMENT")
print("=" * 75)

print("""
Enter the health information you know.
You can press ENTER to skip anything you don't know.

The model will automatically handle missing information.
""")

# ============================================================
# 1. Helper functions
# ============================================================

def get_number(question, unit=""):
    while True:
        value = input(f"{question}{' (' + unit + ')' if unit else ''}: ").strip()

        if value == "":
            return np.nan

        try:
            return float(value)
        except ValueError:
            print("Please enter a number, or press ENTER to skip.")


def get_text(question):
    return input(f"{question}: ").strip()


def normalize_category(value):
    """
    Convert user-friendly categorical answers to the
    string representation used by the trained encoder.
    """

    if value == "":
        return np.nan

    v = value.strip().lower()

    # Gender
    gender_map = {
        "male": "1.0",
        "m": "1.0",
        "man": "1.0",
        "female": "2.0",
        "f": "2.0",
        "woman": "2.0"
    }

    if v in gender_map:
        return gender_map[v]

    # Yes / No
    yes_map = {
        "yes": "1.0",
        "y": "1.0",
        "no": "2.0",
        "n": "2.0"
    }

    # Only use this where the question specifically expects
    # NHANES yes/no coding.

    return value


def get_yes_no(question, yes_value="1.0", no_value="2.0"):
    while True:
        value = input(
            f"{question} [Yes/No, ENTER to skip]: "
        ).strip().lower()

        if value == "":
            return np.nan

        if value in ["yes", "y"]:
            return yes_value

        if value in ["no", "n"]:
            return no_value

        print("Please enter Yes or No.")


# ============================================================
# 2. USER INFORMATION
# ============================================================

user = {}

print("\n" + "-" * 75)
print("1. BASIC INFORMATION")
print("-" * 75)

user["RIDAGEYR"] = get_number(
    "What is your age?",
    "years"
)

# Gender
while True:
    gender = input(
        "What is your sex? [Male/Female, ENTER to skip]: "
    ).strip().lower()

    if gender == "":
        user["RIAGENDR"] = np.nan
        break

    if gender in ["male", "m", "man"]:
        user["RIAGENDR"] = "1.0"
        break

    if gender in ["female", "f", "woman"]:
        user["RIAGENDR"] = "2.0"
        break

    print("Please enter Male or Female.")


# ============================================================
# 3. BLOOD / METABOLIC HEALTH
# ============================================================

print("\n" + "-" * 75)
print("2. BLOOD & METABOLIC HEALTH")
print("-" * 75)

print("""
These values normally come from a blood test.
If you don't have a recent blood report, simply press ENTER.
""")

user["LBXGH"] = get_number(
    "HbA1c",
    "%"
)

user["LBXGLU"] = get_number(
    "Blood glucose",
    "mg/dL"
)

user["LBXIN"] = get_number(
    "Fasting insulin",
    "µU/mL"
)

user["LBDHDD"] = get_number(
    "HDL cholesterol (good cholesterol)",
    "mg/dL"
)

user["LBXTR"] = get_number(
    "Triglycerides",
    "mg/dL"
)

user["LBDLDL"] = get_number(
    "LDL cholesterol (bad cholesterol)",
    "mg/dL"
)

user["LBXAPB"] = get_number(
    "Apolipoprotein B (ApoB)",
    "mg/dL"
)

user["LBXTC"] = get_number(
    "Total cholesterol",
    "mg/dL"
)


# ============================================================
# 4. INCOME / SOCIOECONOMIC INFORMATION
# ============================================================

print("\n" + "-" * 75)
print("3. GENERAL BACKGROUND")
print("-" * 75)

print("""
These variables are part of the dataset's demographic context.
You may skip them if you don't know them.
""")

user["INDHHINC"] = get_number(
    "Approximate annual household income",
    "dataset income category"
)

user["INDFMPIR"] = get_number(
    "Household income relative to the poverty level",
    "ratio"
)


# ============================================================
# 5. EDUCATION
# ============================================================

print("\nEducation level:")

education = input("""
Choose one:
1 = Less than 9th grade
2 = 9th–11th grade
3 = High school graduate
4 = Some college / Associate degree
5 = College graduate or higher
ENTER = Skip

Your choice:
""").strip()

education_map = {
    "1": "1.0",
    "2": "2.0",
    "3": "3.0",
    "4": "4.0",
    "5": "5.0"
}

user["DMDEDUC2"] = education_map.get(
    education,
    np.nan
)


# ============================================================
# 6. MARITAL STATUS
# ============================================================

print("\nMarital status:")

marital = input("""
Choose one:
1 = Married
2 = Widowed
3 = Divorced
4 = Separated
5 = Never married
6 = Living with partner
ENTER = Skip

Your choice:
""").strip()

marital_map = {
    "1": "1.0",
    "2": "2.0",
    "3": "3.0",
    "4": "4.0",
    "5": "5.0",
    "6": "6.0"
}

user["DMDMARTL"] = marital_map.get(
    marital,
    np.nan
)


# ============================================================
# 7. ETHNICITY
# ============================================================

print("\nEthnicity / race:")

race = input("""
Choose one:
1 = Mexican American
2 = Other Hispanic
3 = Non-Hispanic White
4 = Non-Hispanic Black
6 = Non-Hispanic Asian
7 = Other / Multi-racial
ENTER = Skip

Your choice:
""").strip()

race_map = {
    "1": "1.0",
    "2": "2.0",
    "3": "3.0",
    "4": "4.0",
    "6": "6.0",
    "7": "7.0"
}

user["RIDRETH3"] = race_map.get(
    race,
    np.nan
)


# ============================================================
# 8. PHYSICAL ACTIVITY
# ============================================================

print("\n" + "-" * 75)
print("4. PHYSICAL ACTIVITY")
print("-" * 75)

print("""
Tell the model about your normal physical activity.
Press ENTER for anything you don't know.
""")


# Vigorous work
user["PAQ605"] = get_yes_no(
    "Do you do vigorous physical activity at work?"
)

user["PAQ610"] = get_number(
    "How many days per week do you do vigorous activity at work?",
    "days/week"
)

user["PAD615"] = get_number(
    "On those days, how long is the vigorous activity?",
    "minutes/day"
)


# Moderate work
user["PAQ620"] = get_yes_no(
    "Do you do moderate physical activity at work?"
)

user["PAQ625"] = get_number(
    "How many days per week do you do moderate activity at work?",
    "days/week"
)

user["PAD630"] = get_number(
    "On those days, how long is the moderate activity?",
    "minutes/day"
)


# Vigorous recreation
user["PAQ635"] = get_yes_no(
    "Do you do vigorous exercise or sports in your free time?"
)

user["PAQ640"] = get_number(
    "How many days per week do you do vigorous exercise?",
    "days/week"
)

user["PAD645"] = get_number(
    "On those days, how long is the vigorous exercise?",
    "minutes/day"
)


# Moderate recreation
user["PAQ650"] = get_yes_no(
    "Do you do moderate exercise or sports in your free time?"
)

user["PAQ655"] = get_number(
    "How many days per week do you do moderate exercise?",
    "days/week"
)

user["PAD660"] = get_number(
    "On those days, how long is the moderate exercise?",
    "minutes/day"
)


# Walking / cycling
user["PAQ665"] = get_yes_no(
    "Do you walk or cycle for transportation?"
)

user["PAQ670"] = get_number(
    "How many days per week do you walk or cycle?",
    "days/week"
)

user["PAD675"] = get_number(
    "On those days, how long do you walk or cycle?",
    "minutes/day"
)

user["PAD680"] = get_number(
    "How much time do you usually spend sitting?",
    "minutes/day"
)


# ============================================================
# 9. SLEEP
# ============================================================

print("\n" + "-" * 75)
print("5. SLEEP")
print("-" * 75)

sleep_hours = get_number(
    "How many hours do you usually sleep per night?",
    "hours"
)

sleep_minutes = get_number(
    "Additional minutes of sleep per night",
    "minutes"
)

# Convert into the dataset representation
user["SLD010H"] = sleep_hours
user["SLD020M"] = sleep_minutes

user["SLQ030"] = get_number(
    "How often do you have trouble falling asleep?",
    "dataset scale"
)

user["SLQ040"] = get_number(
    "How often do you wake up during the night?",
    "dataset scale"
)

user["SLQ050"] = get_yes_no(
    "Has a doctor ever told you that you have a sleep disorder?"
)

user["SLQ060"] = get_yes_no(
    "Has a doctor ever told you that you have sleep apnea?"
)

user["SLQ120"] = get_yes_no(
    "Do you snore or have breathing problems during sleep?"
)


# ============================================================
# 10. SMOKING
# ============================================================

print("\n" + "-" * 75)
print("6. SMOKING / TOBACCO")
print("-" * 75)

user["SMQ020"] = get_yes_no(
    "Have you smoked at least 100 cigarettes in your lifetime?"
)

user["SMD030"] = get_number(
    "At what age did you start smoking?",
    "years old"
)

user["SMQ040"] = get_number(
    "Do you currently smoke cigarettes?",
    "dataset scale"
)

user["SMQ050Q"] = get_number(
    "How many cigarettes do you usually smoke?",
    "cigarettes"
)

user["SMQ050U"] = get_number(
    "How often do you smoke?",
    "dataset frequency"
)

user["SMD055"] = get_number(
    "How long have you been smoking?",
    "years"
)

user["SMD057"] = get_number(
    "Age when you stopped smoking, if applicable",
    "years old"
)


# ============================================================
# 11. ALCOHOL
# ============================================================

print("\n" + "-" * 75)
print("7. ALCOHOL")
print("-" * 75)

user["ALQ101"] = get_yes_no(
    "Have you had at least one alcoholic drink in your lifetime?"
)

user["ALQ110"] = get_yes_no(
    "Have you had an alcoholic drink in the past 12 months?"
)

user["ALQ120Q"] = get_number(
    "How often do you drink alcohol?",
    "times"
)

user["ALQ120U"] = get_number(
    "Alcohol drinking frequency unit",
    "dataset code"
)

user["ALQ130"] = get_number(
    "On days when you drink, how many drinks do you usually have?",
    "drinks/day"
)

user["ALQ140Q"] = get_number(
    "How often do you have 4 or more drinks in a day?",
    "times"
)

user["ALQ140U"] = get_number(
    "Frequency unit for 4+ drinks",
    "dataset code"
)

user["ALQ150"] = get_number(
    "How often do you have 5 or more drinks in a day?",
    "times"
)

user["ALQ151"] = get_number(
    "How often do you have 8 or more drinks in a day?",
    "times"
)


# ============================================================
# 12. NUTRITION
# ============================================================

print("\n" + "-" * 75)
print("8. DAILY FOOD & NUTRITION")
print("-" * 75)

print("""
These values represent your approximate food intake for one day.

If you don't have a nutrition tracker or don't know the values,
press ENTER.

You can enter the values from apps such as a food diary,
nutrition tracker, or diet plan.
""")

nutrition_questions = {

    "DR1TKCAL":
        ("Daily calories", "kcal/day"),

    "DR1TPROT":
        ("Protein", "g/day"),

    "DR1TCARB":
        ("Carbohydrates", "g/day"),

    "DR1TSUGR":
        ("Total sugar", "g/day"),

    "DR1TFIBE":
        ("Dietary fiber", "g/day"),

    "DR1TTFAT":
        ("Total fat", "g/day"),

    "DR1TSFAT":
        ("Saturated fat", "g/day"),

    "DR1TMFAT":
        ("Monounsaturated fat", "g/day"),

    "DR1TPFAT":
        ("Polyunsaturated fat", "g/day"),

    "DR1TCHOL":
        ("Dietary cholesterol", "mg/day"),

    "DR1TATOC":
        ("Vitamin E", "mg/day"),

    "DR1TRET":
        ("Retinol / preformed vitamin A", "µg/day"),

    "DR1TVARA":
        ("Vitamin A", "RAE/day"),

    "DR1TACAR":
        ("Alpha-carotene", "µg/day"),

    "DR1TBCAR":
        ("Beta-carotene", "µg/day"),

    "DR1TLYCO":
        ("Lycopene", "µg/day"),

    "DR1TLZ":
        ("Lutein + zeaxanthin", "µg/day"),

    "DR1TVB1":
        ("Vitamin B1 (thiamin)", "mg/day"),

    "DR1TVB2":
        ("Vitamin B2 (riboflavin)", "mg/day"),

    "DR1TNIAC":
        ("Niacin / Vitamin B3", "mg/day"),

    "DR1TVB6":
        ("Vitamin B6", "mg/day"),

    "DR1TFOLA":
        ("Folate", "µg/day"),

    "DR1TFA":
        ("Folic acid", "µg/day"),

    "DR1TFF":
        ("Food folate", "µg/day"),

    "DR1TFDFE":
        ("Dietary folate equivalents", "µg/day"),

    "DR1TVB12":
        ("Vitamin B12", "µg/day"),

    "DR1TVC":
        ("Vitamin C", "mg/day"),

    "DR1TVK":
        ("Vitamin K", "µg/day"),

    "DR1TCALC":
        ("Calcium", "mg/day"),

    "DR1TPHOS":
        ("Phosphorus", "mg/day"),

    "DR1TMAGN":
        ("Magnesium", "mg/day"),

    "DR1TIRON":
        ("Iron", "mg/day"),

    "DR1TZINC":
        ("Zinc", "mg/day"),

    "DR1TCOPP":
        ("Copper", "mg/day"),

    "DR1TSODI":
        ("Sodium", "mg/day"),

    "DR1TPOTA":
        ("Potassium", "mg/day"),

    "DR1TSELE":
        ("Selenium", "µg/day"),

    "DR1TCAFF":
        ("Caffeine", "mg/day"),

    "DR1TALCO":
        ("Alcohol from food/drinks", "g/day")
}

for col in NUTRITION_COLS:

    question, unit = nutrition_questions.get(
        col,
        (col, "")
    )

    user[col] = get_number(
        question,
        unit
    )


# ============================================================
# 13. CREATE DATAFRAME
# ============================================================

input_df = pd.DataFrame([user])


# Ensure all numerical features exist
all_numeric_cols = (
    BIOMARKER_COLS +
    NUTRITION_COLS +
    LIFESTYLE_COLS +
    DEMO_NUM_COLS
)

for col in all_numeric_cols:

    if col not in input_df.columns:
        input_df[col] = np.nan

    input_df[col] = pd.to_numeric(
        input_df[col],
        errors="coerce"
    )


# Ensure categorical columns exist
for col in DEMO_CAT_COLS:

    if col not in input_df.columns:
        input_df[col] = np.nan

    input_df[col] = input_df[col].astype(object)


# ============================================================
# 14. SAME FEATURE ENGINEERING AS TRAINING
# ============================================================

input_df = add_lifestyle_features(input_df)


# ============================================================
# 15. MISSINGNESS MASKS
# ============================================================

bio_missing = (
    input_df[BIOMARKER_COLS]
    .isna()
    .values
    .astype(np.int64)
)

nutri_missing = (
    input_df[NUTRITION_COLS]
    .isna()
    .values
    .astype(np.int64)
)

life_missing = (
    input_df[LIFESTYLE_NUM_COLS]
    .isna()
    .values
    .astype(np.int64)
)


# ============================================================
# 16. PREPROCESS NUMERICAL FEATURES
# ============================================================

bio_processed = prep["bio_scaler"].transform(
    prep["bio_imputer"].transform(
        input_df[BIOMARKER_COLS]
    )
)

nutri_processed = prep["nutri_scaler"].transform(
    prep["nutri_imputer"].transform(
        input_df[NUTRITION_COLS]
    )
)

life_processed = prep["life_scaler"].transform(
    prep["life_imputer"].transform(
        input_df[LIFESTYLE_NUM_COLS]
    )
)


# ============================================================
# 17. PREPROCESS CATEGORICAL FEATURES
# ============================================================

cat_raw = input_df[DEMO_CAT_COLS].copy()

cat_imputed = prep["cat_imputer"].transform(cat_raw)

cat_encoded = prep["cat_encoder"].transform(
    cat_imputed
)

# Handle unknown categories
for j in range(cat_encoded.shape[1]):

    unknown_index = len(
        prep["cat_encoder"].categories_[j]
    )

    cat_encoded[:, j] = np.where(
        cat_encoded[:, j] < 0,
        unknown_index,
        cat_encoded[:, j]
    )

cat_encoded = cat_encoded.astype(np.int64)


# ============================================================
# 18. CREATE MODEL INPUT
# ============================================================

new_input = {

    "bio": torch.tensor(
        bio_processed,
        dtype=torch.float32
    ).to(device),

    "bio_missing": torch.tensor(
        bio_missing,
        dtype=torch.long
    ).to(device),

    "nutri": torch.tensor(
        nutri_processed,
        dtype=torch.float32
    ).to(device),

    "nutri_missing": torch.tensor(
        nutri_missing,
        dtype=torch.long
    ).to(device),

    "life": torch.tensor(
        life_processed,
        dtype=torch.float32
    ).to(device),

    "life_missing": torch.tensor(
        life_missing,
        dtype=torch.long
    ).to(device),

    "life_cat": torch.tensor(
        cat_encoded,
        dtype=torch.long
    ).to(device)
}


# ============================================================
# 19. LOAD BEST TRAINED MODEL
# ============================================================

print("\n" + "-" * 75)
print("RUNNING BioNutriGPT...")
print("-" * 75)

checkpoint = torch.load(
    best_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state"]
)

model.eval()


# ============================================================
# 20. PREDICTION
# ============================================================

with torch.no_grad():

    output = model(new_input)

    probabilities = (
        torch.sigmoid(
            output["logits"]
        )
        .cpu()
        .numpy()[0]
    )

    gate_weights = (
        output["gate_weights"]
        .cpu()
        .numpy()[0]
    )


# ============================================================
# 21. RESULTS
# ============================================================

print("\n")
print("=" * 75)
print("                    YOUR RESULTS")
print("=" * 75)

risk_names = [
    "Type 2 Diabetes Risk",
    "Obesity Risk",
    "Cardiovascular Disease Risk"
]

for name, probability in zip(
    risk_names,
    probabilities
):

    percentage = probability * 100

    if percentage < 30:
        level = "LOW"
    elif percentage < 60:
        level = "MODERATE"
    else:
        level = "HIGH"

    print(
        f"{name:<35} {percentage:6.2f}%   [{level}]"
    )


# ============================================================
# 22. MODEL INPUT COVERAGE
# ============================================================

total_features = (
    len(BIOMARKER_COLS) +
    len(NUTRITION_COLS) +
    len(LIFESTYLE_NUM_COLS)
)

missing_features = (
    int(bio_missing.sum()) +
    int(nutri_missing.sum()) +
    int(life_missing.sum())
)

available_features = total_features - missing_features

coverage = (
    available_features /
    total_features *
    100
)

print("\n" + "-" * 75)
print("INPUT COVERAGE")
print("-" * 75)

print(
    f"You provided information for approximately "
    f"{coverage:.1f}% of the model's numerical features."
)

print(
    f"Missing values: {missing_features}/{total_features}"
)

print(
    "Missing information was automatically estimated using "
    "the training preprocessing pipeline."
)


# ============================================================
# 23. MODALITY CONTRIBUTION
# ============================================================

print("\n" + "-" * 75)
print("WHAT INFORMATION INFLUENCED THE MODEL?")
print("-" * 75)

print(
    f"Blood / Metabolic data : {gate_weights[0] * 100:.2f}%"
)

print(
    f"Nutrition               : {gate_weights[1] * 100:.2f}%"
)

print(
    f"Lifestyle / Demographics: {gate_weights[2] * 100:.2f}%"
)


# ============================================================
# 24. INTERPRETATION
# ============================================================

print("\n" + "=" * 75)
print("IMPORTANT")
print("=" * 75)

print("""
These percentages are predictions produced by your trained
BioNutriGPT model. They are NOT medical diagnoses.

A higher percentage means the model estimates a higher predicted
risk based on the information provided.

Missing information is automatically imputed, so predictions made
with very little personal information should be interpreted with
more caution.
""")

print("=" * 75)

             BioNutriGPT PERSONAL HEALTH ASSESSMENT

Enter the health information you know.
You can press ENTER to skip anything you don't know.

The model will automatically handle missing information.


---------------------------------------------------------------------------
1. BASIC INFORMATION
---------------------------------------------------------------------------
What is your age? (years): 58
What is your sex? [Male/Female, ENTER to skip]: Male

---------------------------------------------------------------------------
2. BLOOD & METABOLIC HEALTH
---------------------------------------------------------------------------

These values normally come from a blood test.
If you don't have a recent blood report, simply press ENTER.

HbA1c (%): 2
Blood glucose (mg/dL): 120
Fasting insulin (µU/mL): 
HDL cholesterol (good cholesterol) (mg/dL): 
Triglycerides (mg/dL): 
LDL cholesterol (bad cholesterol) (mg/dL): 
Apolipoprotein B (ApoB) (mg/dL): 
Total cholesterol (mg/dL): 

-------

## Final output

The notebook implements exactly this stage:

```text
BIOMARKERS ── FT-Transformer ── Z_bio ───────────┐
                                                 │
NUTRITION ──── SAINT ─────────── Z_nutri ────────┼── Cross-Modal Transformer
                                                 │
LIFESTYLE + DEMOGRAPHICS ─ FT-Transformer ─ Z_life┘
                                                        ↓
                                                     Z_fused
                                                        ↓
                                                  Adaptive MoE
                                       ┌────────────┼────────────┐
                                  Metabolic      Nutrition      Lifestyle
                                    Expert        Expert          Expert
                                       └────────────┼────────────┘
                                                    ↓
                                                  Z_MoE
                                                    ↓
                                ┌───────────────────┼───────────────────┐
                                ↓                   ↓                   ↓
                            Diabetes             Obesity               CVD
                              Head                 Head                Head
                                ↓                   ↓                   ↓
                                      Probability + MC-Dropout
                                                ↓
                                   Risk + Uncertainty

Saved artifacts:
- best_bionutrigt_multimodal.pt
- risk_predictions_with_uncertainty.csv
- test_embeddings_and_moe_gates.csv
- preprocessing.joblib
- training_history.csv
- architecture.json
